### Step 1. Load yolo model to predict label and bounding box.

In [1]:
from ultralytics import YOLO
from pathlib import Path

In [2]:
model_weight = "weights/yolo/best_yolo11m_pencilbag.pt"
images_path = Path("datasets/robot-detection/data/images/pencilbag_task/")
boundingbox_path = Path("datasets/robot-detection/boundingbox-label/pencilbag_task/")


valid_suffixes = {".png", ".jpg", ".jpeg", ".bmp"}
images = [f for f in images_path.iterdir() if f.is_file() and f.suffix.lower() in valid_suffixes]
    
print(images)


[PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_88.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_118.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_189.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_252.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_230.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_284.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_55.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_233.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_92.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_185.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_130.png'), PosixPath('datasets/robot-detection/data/images/pencilbag_task/pencilbag_5.png

### load model and enum predict per image

In [3]:
### build conversations and jsonl type

def build_jsonl_conversation_type(label_list,bbx_list,image_path):
    ##
    ## label_list : truth labels for traget like : pencilbag,pen .
    ## bbx_list : bounding box target list,like : [x1,y1,x1,y1] 
    ## image_path

    # target jsonl:
    # {"conversations": [
    # {"from": "human", "value": "Locate all the instances that matches the following description: car</c>person</c>bicycle."}, 
    # {"from": "gpt", "value": "<ref>car</ref><box><120><200><450><500></box><ref>person</ref><box><50><100><200><600></box><ref>person</ref><box><300><150><420><580></box>"}], 
    # "image": "coco/train2017/000001.jpg"
    # }
    
    human_description = ""
    gpt_description = ""

    for index,label in enumerate(label_list):
        if index == len(label_list) - 1:
            human_description = human_description + label
            
        else:  
            human_description = human_description + label + "</c>"
        bbx_string = ""
        for i in range(4):
            bbx_string = bbx_string + "<"+str(bbx_list[index][i])+">"
        gpt_description = gpt_description + "<ref>"+label+"</ref><box>" + bbx_string + "</box>"


    target_jsonl = {"conversations": [
    {"from": "human", 
     "value": "Locate all the instances that matches the following description: "+f"{human_description}"}, 
    {"from": "gpt", 
     "value": gpt_description}], 
    "image": image_path.as_posix()
    }
    return target_jsonl
        
    
    

In [4]:
from tqdm import tqdm
import json

write_jsonl = "datasets/robot-detection/data/annotations/detection.jsonl"

model = YOLO(model_weight)

label_map = model.names

img_width = 640
img_height = 480

with open(write_jsonl,"a") as f:
    for image_path in tqdm(images, desc="processing pred:"):
    
        results = model(image_path)
        result = results[0]
        boxes = result.boxes
    
        bbx_list = []
        label_list = []
    
        if len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()    
            conf = boxes.conf.cpu().numpy()    
            cls = boxes.cls.cpu().numpy()     
    
            for box, score, class_id in zip(xyxy, conf, cls):
                x1, y1, x2, y2 = box
                if score <=0.6:
                    continue
                #print("pred res:",end="")
                #print(box,score,class_id)
                class_id = int(class_id)
                get_label = label_map[class_id]
    
                label_list.append(get_label)
    
                ## convert box range - norm 
                x1 = int(x1 / img_width  * 1000)
                y1 = int(y1 / img_height * 1000)
                x2 = int(x2 / img_width  * 1000)
                y2 = int(y2 / img_height * 1000)
    
                bbx_list.append((x1,y1,x2,y2))
                
        if len(label_list) > 0:
            
            get_jsonl = build_jsonl_conversation_type(label_list,bbx_list,image_path)
            #print(get_jsonl)
            
            f.write(json.dumps(get_jsonl) + "\n")
            
            

processing pred::   0%|                                                                                                                                       | 0/311 [00:00<?, ?it/s]


image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_88.png: 480x640 1 pencilbag, 1 pig zipper, 1 simple zipper, 2 pens, 30.2ms
Speed: 1.2ms preprocess, 30.2ms inference, 16.5ms postprocess per image at shape (1, 3, 480, 640)


processing pred::   0%|▍                                                                                                                              | 1/311 [00:00<04:00,  1.29it/s]


image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_118.png: 480x640 1 pencilbag, 1 pig zipper, 1 simple zipper, 1 pen, 9.7ms
Speed: 1.1ms preprocess, 9.7ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_189.png: 480x640 1 pencilbag, 1 pig zipper, 1 simple zipper, 10.0ms
Speed: 1.4ms preprocess, 10.0ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_252.png: 480x640 1 pencilbag, 1 pig zipper, 1 simple zipper, 10.0ms
Speed: 0.9ms preprocess, 10.0ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_230.png: 480x640 1 pencilbag, 1 

processing pred::   2%|██▍                                                                                                                            | 6/311 [00:00<00:34,  8.85it/s]

...

processing pred::  98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏  | 304/311 [00:06<00:00, 48.02it/s]


image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_33.png: 480x640 1 pencilbag, 1 pig zipper, 1 simple zipper, 1 pen, 10.3ms
Speed: 0.6ms preprocess, 10.3ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_209.png: 480x640 1 pencilbag, 1 pig zipper, 1 simple zipper, 1 pen, 10.2ms
Speed: 0.6ms preprocess, 10.2ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_216.png: 480x640 2 pencilbags, 1 pig zipper, 1 simple zipper, 4 pens, 10.7ms
Speed: 0.8ms preprocess, 10.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_75.png: 480x640

processing pred:: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 310/311 [00:07<00:00, 48.89it/s]


image 1/1 /home/yy/yy/github/Locateanything-LoRA-case/datasets/robot-detection/data/images/pencilbag_task/pencilbag_290.png: 480x640 1 pencilbag, 1 pig zipper, 1 simple zipper, 1 pen, 10.4ms
Speed: 0.8ms preprocess, 10.4ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)


processing pred:: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:07<00:00, 44.06it/s]


### Step 2. use pred label and bounding box to make "conversation" json type

### Step 3. make JSONL and Recipe json file

### Add giftbox task 

In [5]:

model_weight = "weights/yolo/best_yolo11s_giftbox.pt"
images_path = Path("datasets/robot-detection/data/images/giftbox_task/")
#boundingbox_path = Path("datasets/robot-detection/boundingbox-label/pencilbag_task/")


valid_suffixes = {".png", ".jpg", ".jpeg", ".bmp"}
images = [f for f in images_path.iterdir() if f.is_file() and f.suffix.lower() in valid_suffixes]
    
print(images)


[PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_96.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_99.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_33.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_50.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_153.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_22.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_169.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_6.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_220.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_140.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_5.png'), PosixPath('datasets/robot-detection/data/images/giftbox_task/giftbox_44.png'), PosixPath('datasets/robot-detection/data/images/g

In [6]:
from tqdm import tqdm
import json

write_jsonl = "datasets/robot-detection/data/annotations/detection.jsonl"

model = YOLO(model_weight)

label_map = model.names

img_width = 640
img_height = 480

with open(write_jsonl,"a") as f:
    for image_path in tqdm(images, desc="processing pred:"):
    
        results = model(image_path,verbose=False)
        result = results[0]
        boxes = result.boxes
    
        bbx_list = []
        label_list = []
    
        if len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()    
            conf = boxes.conf.cpu().numpy()    
            cls = boxes.cls.cpu().numpy()     
    
            for box, score, class_id in zip(xyxy, conf, cls):
                x1, y1, x2, y2 = box
                if score <=0.6:
                    continue
                #print("pred res:",end="")
                #print(box,score,class_id)
                class_id = int(class_id)
                get_label = label_map[class_id]
    
                label_list.append(get_label)
    
                ## convert box range - norm 
                x1 = int(x1 / img_width  * 1000)
                y1 = int(y1 / img_height * 1000)
                x2 = int(x2 / img_width  * 1000)
                y2 = int(y2 / img_height * 1000)
    
                bbx_list.append((x1,y1,x2,y2))
                
        if len(label_list) > 0:
            
            get_jsonl = build_jsonl_conversation_type(label_list,bbx_list,image_path)
            #print(get_jsonl)
            
            f.write(json.dumps(get_jsonl) + "\n")
            
            

processing pred:: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 228/228 [00:01<00:00, 115.84it/s]


### Package fun

In [7]:
from ultralytics import YOLO
from pathlib import Path
from tqdm import tqdm
import json

def gengerate_datasets_jsonl(model_weight,images_path,write_jsonl,img_width,img_height):

    ## defult write jsonl file method is "a"
    ## model_weight mean yolo pt path
    ## images_path is process data image path
    ## image W,H : img_width,img_height


    
    valid_suffixes = {".png", ".jpg", ".jpeg", ".bmp"}
    images = [f for f in images_path.iterdir() if f.is_file() and f.suffix.lower() in valid_suffixes]

    ## load model
    model = YOLO(model_weight)
    ## get label map , pt have save the label map
    label_map = model.names
    
    
    with open(write_jsonl,"a") as f:
        for image_path in tqdm(images, desc="processing pred:"):
        
            results = model(image_path,verbose=False)
            result = results[0]
            boxes = result.boxes
        
            bbx_list = []
            label_list = []
        
            if len(boxes) > 0:
                xyxy = boxes.xyxy.cpu().numpy()    
                conf = boxes.conf.cpu().numpy()    
                cls = boxes.cls.cpu().numpy()     
        
                for box, score, class_id in zip(xyxy, conf, cls):
                    x1, y1, x2, y2 = box
                    if score <=0.6:
                        continue
                    #print("pred res:",end="")
                    #print(box,score,class_id)
                    class_id = int(class_id)
                    get_label = label_map[class_id]
        
                    label_list.append(get_label)
        
                    ## convert box range - norm 
                    x1 = int(x1 / img_width  * 1000)
                    y1 = int(y1 / img_height * 1000)
                    x2 = int(x2 / img_width  * 1000)
                    y2 = int(y2 / img_height * 1000)
        
                    bbx_list.append((x1,y1,x2,y2))
                    
            if len(label_list) > 0:
                
                get_jsonl = build_jsonl_conversation_type(label_list,bbx_list,image_path)
                #print(get_jsonl)
                
                f.write(json.dumps(get_jsonl) + "\n")
    
    

In [8]:
### build conversations and jsonl type

def build_jsonl_conversation_type(label_list,bbx_list,image_path):
    ##
    ## label_list : truth labels for traget like : pencilbag,pen .
    ## bbx_list : bounding box target list,like : [x1,y1,x1,y1] 
    ## image_path

    # target jsonl:
    # {"conversations": [
    # {"from": "human", "value": "Locate all the instances that matches the following description: car</c>person</c>bicycle."}, 
    # {"from": "gpt", "value": "<ref>car</ref><box><120><200><450><500></box><ref>person</ref><box><50><100><200><600></box><ref>person</ref><box><300><150><420><580></box>"}], 
    # "image": "coco/train2017/000001.jpg"
    # }
    
    human_description = ""
    gpt_description = ""

    for index,label in enumerate(label_list):
        if index == len(label_list) - 1:
            human_description = human_description + label
            
        else:  
            human_description = human_description + label + "</c>"
        bbx_string = ""
        for i in range(4):
            bbx_string = bbx_string + "<"+str(bbx_list[index][i])+">"
        gpt_description = gpt_description + "<ref>"+label+"</ref><box>" + bbx_string + "</box>"


    target_jsonl = {"conversations": [
    {"from": "human", 
     "value": "Locate all the instances that matches the following description: "+f"{human_description}"}, 
    {"from": "gpt", 
     "value": gpt_description}], 
    "image": image_path.as_posix()
    }
    return target_jsonl
        
    
    

In [9]:
## all args :
# defult 
write_jsonl = "datasets/robot-detection/data/annotations/detection.jsonl"
img_width = 640
img_height = 480


In [10]:
model_weight = "weights/yolo/best_yolo11m_dir.pt"
images_path = Path("datasets/robot-detection/data/images/dictionary_task/")


gengerate_datasets_jsonl(model_weight,images_path,write_jsonl,img_width,img_height)

processing pred:: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:02<00:00, 47.78it/s]


In [11]:
model_weight = "weights/yolo/best_yolo11m_beaker.pt"
images_path = Path("datasets/robot-detection/data/images/beaker_task/")


gengerate_datasets_jsonl(model_weight,images_path,write_jsonl,img_width,img_height)

processing pred:: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 68/68 [00:01<00:00, 45.71it/s]
